In [3]:
import os
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# --------------------------
# 설정
# --------------------------
MODEL_DIR = "./my_kobert_best_model_full"
INPUT_CSV = "movie_long_with_emotions_full.csv"
OUTPUT_CSV = "movie_long_with_emotions and polarity_full.csv"
TEXT_COL = "감상평"
MAX_LEN = 64
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROUND = 6   # 저장 시 소수점 자리(원하면 None)

# --------------------------
# 유틸: CSV 안전 로더 (여러 인코딩 시도)
# --------------------------
def load_csv_safely(path, **kwargs):
    tried = []
    for enc in ["utf-8", "utf-8-sig", "cp949", "euc-kr", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            print(f"[INFO] CSV loaded with encoding = {enc}")
            return df
        except UnicodeDecodeError as e:
            tried.append(enc)
    # 마지막 방어막: 깨지는 문자는 치환
    try:
        df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)
        print("[WARN] Fallback: utf-8 with encoding_errors='replace' (문자 일부 치환 가능)")
        return df
    except Exception as e:
        raise RuntimeError(f"CSV 인코딩 자동 감지 실패. 시도한 인코딩: {tried}") from e

# --------------------------
# 체크
# --------------------------
if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(f"모델 폴더가 없습니다: {MODEL_DIR}")
if not os.path.isfile(INPUT_CSV):
    raise FileNotFoundError(f"입력 CSV가 없습니다: {INPUT_CSV}")

# --------------------------
# 로드
# --------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

# --------------------------
# 데이터셋 / 로더
# --------------------------
class ReviewDS(Dataset):
    def __init__(self, df, text_col, tokenizer, max_len):
        if text_col not in df.columns:
            raise ValueError(f"CSV에 '{text_col}' 컬럼이 없습니다. 실제 컬럼명 확인: {list(df.columns)[:10]}")
        self.texts = df[text_col].astype(str).fillna("")
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer.encode_plus(
            self.texts.iloc[idx],
            add_special_tokens=True, max_length=self.max_len,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }

# ---- CSV 읽기 (여기만 교체됨)
df = load_csv_safely(INPUT_CSV)

dataset = ReviewDS(df, TEXT_COL, tokenizer, MAX_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# --------------------------
# 추론
# --------------------------
prob_positives = []
pred_texts = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Scoring"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)     # [B, 2]
        pos = probs[:, 1].detach().cpu().numpy()     # 긍정(1) 확률
        prob_positives.extend(pos.tolist())
        pred_texts.extend(np.where(pos >= 0.5, "긍정", "부정").tolist())

# --------------------------
# 점수 변환 (0~1) -> (-1~+1)
# --------------------------
prob_positives = np.array(prob_positives, dtype=np.float32)
polarity_pm1 = (prob_positives * 2.0) - 1.0         # -1 ~ +1

if ROUND is not None:
    prob_positives = np.round(prob_positives, ROUND)
    polarity_pm1 = np.round(polarity_pm1, ROUND)

# --------------------------
# 저장
# --------------------------
df_out = df.copy()
df_out["polarity_prob_0_1"] = prob_positives  # 0~1
df_out["polarity_pm1"] = polarity_pm1         # -1~+1
df_out["pred_label"] = pred_texts             # 옵션

# 출력은 가급적 UTF-8-SIG로 저장(엑셀 호환)
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"완료. '{OUTPUT_CSV}' 로 저장했습니다. (rows={len(df_out)})")


[INFO] CSV loaded with encoding = cp949


Scoring: 100%|██████████| 1167/1167 [10:19<00:00,  1.88it/s]


완료. 'movie_long_with_emotions and polarity_full.csv' 로 저장했습니다. (rows=74634)
